# 00 · Join UBIGEO y carga de fuentes de datos

Notebook de la Etapa 0 del pipeline: antes de tocar una sola imagen, se cargan y verifican todas las fuentes de datos y se construye la tabla maestra unida por UBIGEO.

## Setup

In [1]:
import pandas as pd
import geopandas as gpd
import requests

pd.set_option("display.max_columns", 50)


# Polígonos y ubicación geográfica

## Límite distrital INEI 2025 (shapefile)

Fuente: https://www.geogpsperu.com/2019/05/limite-distrital-actualizado-inei.html

Descarga manual (no tiene link directo de descarga)

In [ ]:
# Ruta local al shapefile ya descargado
ruta_limite_distrital = "../data/raw/Limite Distrital INEI 2025 CPV/Limite Distrital INEI 2025 CPV.shp"

limite_distrital = gpd.read_file(ruta_limite_distrital)
print(limite_distrital.shape)
limite_distrital.head()


(1891, 10)


,UBIGEO,CCDD,CCPP,CCDI,DEPARTAMEN,PROVINCIA,DISTRITO,OBJECTID,ESRI_OID,geometry
0,010101,01,01,01,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,1.0,5.0,"POLYGON ((-77.8858 -6.1778, -77.88323 -6.17846..."
1,010102,01,01,02,AMAZONAS,CHACHAPOYAS,ASUNCION,2.0,6.0,"POLYGON ((-77.74482 -5.94497, -77.74482 -5.945..."
2,010103,01,01,03,AMAZONAS,CHACHAPOYAS,BALSAS,3.0,7.0,"POLYGON ((-77.9358 -6.69039, -77.93531 -6.6909..."
3,010104,01,01,04,AMAZONAS,CHACHAPOYAS,CHETO,4.0,8.0,"POLYGON ((-77.71486 -6.24598, -77.71485 -6.245..."
4,010105,01,01,05,AMAZONAS,CHACHAPOYAS,CHILIQUIN,5.0,9.0,"POLYGON ((-77.77405 -5.99598, -77.77328 -5.996..."


## Tabla de UBIGEO (crosswalk departamento-provincia-distrito)

Fuente (CSV público, se puede leer directo desde la URL):
https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv

In [ ]:
url_ubigeo = "https://raw.githubusercontent.com/jmcastagnetto/ubigeo-peru-aumentado/main/ubigeo_distrito.csv"

ubigeo = pd.read_csv(url_ubigeo, dtype={"inei": str, "reniec": str})

print(ubigeo.shape)
ubigeo.head()

URLError: <urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1028)>

Como tenemos diferente cantidad de filas, verificaremos cuáles sonla que faltan en la 1ra

In [ ]:
fila_nan = ubigeo[ubigeo["inei"].isna()]
fila_nan

,inei,reniec,departamento,provincia,distrito,region,macroregion_inei,macroregion_minsa,iso_3166_2,fips,capital,superficie,pob_densidad_2020,altitude,latitude,longitude,indice_vulnerabilidad_alimentaria,idh_2019,pct_pobreza_total,pct_pobreza_extrema
1892,NaN,170107,MOQUEGUA,MARISCAL NIETO,SAN ANTONIO,MOQUEGUA,SUR,MACROREGION SUR,PE-MOQ,18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Quitar la fila con inei vacío (San Antonio, Moquegua - dato incompleto)
ubigeo = ubigeo[ubigeo["inei"].notna()].copy()
print("Distritos después de quitar el nan:", ubigeo.shape[0])

Distritos después de quitar el nan: 1892


In [ ]:
# Los que están en el shapefile pero no en la tabla UBIGEO
print("--- En shapefile, no en tabla ---")
print(limite_distrital[limite_distrital["UBIGEO"].isin(["180107", "130112"])][["UBIGEO", "DEPARTAMEN", "PROVINCIA", "DISTRITO"]])

print()

# Los que están en la tabla UBIGEO pero no en el shapefile
print("--- En tabla, no en shapefile ---")
print(ubigeo[ubigeo["inei"].isin(["160109", "150144", "160114"])][["inei", "departamento", "provincia", "distrito"]])

--- En shapefile, no en tabla ---
      UBIGEO   DEPARTAMEN       PROVINCIA       DISTRITO
1182  130112  LA LIBERTAD        TRUJILLO  ALTO TRUJILLO
1534  180107     MOQUEGUA  MARISCAL NIETO    SAN ANTONIO

--- En tabla, no en shapefile ---
        inei departamento provincia                 distrito
1335  150144         LIMA      LIMA  SANTA MARIA DE HUACHIPA
1472  160109       LORETO    MAYNAS                 PUTUMAYO
1476  160114       LORETO    MAYNAS  TENIENTE MANUEL CLAVERO


In [ ]:
ubigeos_finales = set(limite_distrital["UBIGEO"].astype(str)) & set(ubigeo["inei"].astype(str))
print("Distritos finales para el análisis:", len(ubigeos_finales))

limite_distrital = limite_distrital[limite_distrital["UBIGEO"].isin(ubigeos_finales)].copy()
ubigeo = ubigeo[ubigeo["inei"].isin(ubigeos_finales)].copy()

print("Shapefile final:", limite_distrital.shape[0])
print("Tabla ubigeo final:", ubigeo.shape[0])

Distritos finales para el análisis: 1889
Shapefile final: 1889
Tabla ubigeo final: 1889
